# CNNs: BatchNorm per-channel broadcasting

**Solution notebook — Delta Drills #459**

Run the cells top-to-bottom to see the reference answer execute.


## Problem

BatchNorm's per-channel vectors (weight, bias, running mean, running var — each length C) cannot broadcast against a (B, C, H, W) tensor as-is, because broadcasting aligns TRAILING dims. Write solve(x, weight, bias, mean, var): reshape each vector so the channel axis lands on dim 1 with singleton batch/spatial dims, then compute normalize-scale-shift: (x − mean)/sqrt(var) * weight + bias. Return the result tensor.


<details><summary>💡 Hint (click to reveal)</summary>

Reshape all four vectors to (1, C, 1, 1); then (x−m)/√v · w + b broadcasts cleanly.

</details>


In [ ]:
%pip install -q numpy torch --index-url https://download.pytorch.org/whl/cpu

## Reference solution


In [ ]:
import torch

def solve(x, weight, bias, mean, var):
    C = x.shape[1]
    w = weight.reshape(1, C, 1, 1)
    b = bias.reshape(1, C, 1, 1)
    m = mean.reshape(1, C, 1, 1)
    v = var.reshape(1, C, 1, 1)
    return (x - m) / torch.sqrt(v) * w + b

x = torch.arange(24, dtype=torch.float32).reshape(2, 3, 2, 2)
weight = torch.tensor([1.0, 2.0, 0.5])
bias = torch.tensor([0.0, 1.0, -1.0])
mean = torch.tensor([0.0, 4.0, 8.0])
var = torch.tensor([1.0, 4.0, 16.0])
print(solve(x, weight, bias, mean, var))


## Why this works

Broadcasting aligns trailing dims, so a length-C vector lines up with W, not C. Reshaping to (1, C, 1, 1) plants each vector on the channel axis with singletons elsewhere; then the whole normalize-scale-shift is four elementwise broadcast ops — exactly what BatchNorm does internally.
